In [ ]:
!nvidia-smi     # confirm: Tesla T4, ~15 GB VRAM

In [ ]:
# The CUDA architecture flag (75 = T4) makes the compile much faster.
!CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75" pip install llama-cpp-python --no-cache-dir
!pip install -q huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 MB 80.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.36-py3-none-linux_x86_64.whl size=268942452 sha256=2ecdc37114b9de1996727cd4cb09e925bc7067a8a18ee4b7eb542867ede46100
  Stored in directory: /tmp/pip-ephem-wheel-cache-dv75zuz4/wheels/e5/bf/ad/f5f50f854633df40f54d89e4e5271e987476036203819f8b4c
Successfully built llama-cpp-python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [ ]:
# GGML (.bin) is dead, so the current format is GGUF.
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename     = "llama-2-13b-chat.Q5_K_M.gguf"   # Q5_K_M ≈ the old q5_1 (5-bit); ~9 GB

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

model_path = hf_hub_download(repo_id=model_name_or_path, filename=model_basename)   # ~9 GB download
print(model_path)

lcpp_llm = Llama(
    model_path=model_path,
    n_threads=2,        # CPU cores
    n_batch=512,        # tune against VRAM
    n_gpu_layers=-1,    # -1 = put ALL layers on the GPU (the notes used 32; 13B has 40, so -1 is simpler)
    n_ctx=2048,         # context window
    verbose=True,       # shows "offloaded ... layers to GPU" in the log
)

In [ ]:
prompt = "Write a MNIST image classification code in keras"

# Llama-2-chat's real prompt format: [INST] <<SYS>> system <</SYS>> user [/INST]
prompt_template = f"""[INST] <<SYS>>
You are a helpful, respectful and honest assistant. Always answer as helpfully as possible.
<</SYS>>

{prompt} [/INST]
"""

response = lcpp_llm(
    prompt=prompt_template,
    max_tokens=256,
    temperature=0.5,
    top_p=0.95,
    repeat_penalty=1.2,
    top_k=150,
    echo=True,          # echo the prompt back
)

print(response["choices"][0]["text"])